In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# SILVER — physical_lojas
# Squad 3 — Arquitetura Medalhao
# Regra: tratamentos e regras tecnicas
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════

# COMMAND ----------



In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

BRONZE_TABLE = "physical_lojas"
BRONZE_PATH  = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "physical_lojas"
SILVER_PATH  = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

KEY_COLUMNS       = ["id_loja"]
SILVER_WRITE_MODE = "overwrite"

SILVER_REQUIRED_COLUMNS = [
    "id_loja",
    "nome_loja",
    "cnpj",
    "cidade_loja",
    "estado_loja",
    "peso_vendas",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
]

UFS_VALIDAS = [
    "AC","AL","AP","AM","BA","CE","DF","ES","GO",
    "MA","MT","MS","MG","PA","PB","PR","PE","PI",
    "RJ","RN","RS","RO","RR","SC","SP","SE","TO"
]

print("Constantes configuradas:")
print(f"   BRONZE_PATH : {BRONZE_PATH}")
print(f"   SILVER_PATH : {SILVER_PATH}")
print(f"   KEY_COLUMNS : {KEY_COLUMNS}")

In [0]:
# configuracoes do ADLS

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler Bronze

df_bronze = read_delta(
    spark        = spark,
    path         = BRONZE_PATH,
    adls_options = adls_options
)

df_bronze.printSchema()
total_bronze = df_bronze.count()
print(f"Total de registros na Bronze: {total_bronze:,}")
display(df_bronze.limit(10))

In [0]:
# validar conversoes antes de aplicar

from pyspark.sql.functions import col, count, when

df_validacao_conv = df_bronze.select(
    count("*").alias("total_linhas"),
    count(
        when(
            col("id_loja").isNotNull() &
            col("id_loja").cast("int").isNull(),
            True
        )
    ).alias("falhas_id_loja"),
    count(
        when(
            col("peso_vendas").isNotNull() &
            col("peso_vendas").cast("int").isNull(),
            True
        )
    ).alias("falhas_peso_vendas"),
)

display(df_validacao_conv)
validacao_conv = df_validacao_conv.collect()[0]

if validacao_conv["falhas_id_loja"] > 0:
    raise Exception(
        f"Existem {validacao_conv['falhas_id_loja']} valores "
        f"de id_loja que nao podem ser convertidos para integer."
    )

print("Validacao OK: conversoes principais podem ser feitas.")

In [0]:
# aplicar transformacoes Silver

from pyspark.sql.functions import (
    col, trim, upper,
    when, regexp_replace,
    current_timestamp, row_number
)
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

# deduplicacao por id_loja
# mantem o registro mais recente
window_dedup = (
    Window
    .partitionBy("id_loja")
    .orderBy(col("bronze_ingested_at").desc_nulls_last())
)

df_silver = (
    df_bronze

    # converter tipos
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
    .withColumn("peso_vendas",
        col("peso_vendas").cast(IntegerType()))

    # CNPJ: remover nao numericos
    .withColumn("cnpj",
        regexp_replace(col("cnpj"), r"[^0-9]", ""))

    # padronizar nome_loja — tratar "nan"
    .withColumn(
        "nome_loja",
        when(
            trim(col("nome_loja")).contains("nan") |
            col("nome_loja").isNull(),
            "Nao Informado"
        ).otherwise(trim(col("nome_loja")))
    )

    # padronizar texto
    .withColumn("cidade_loja", trim(upper(col("cidade_loja"))))
    .withColumn("estado_loja", trim(upper(col("estado_loja"))))

    # deduplicacao
    .withColumn("rn", row_number().over(window_dedup))
    .filter(col("rn") == 1)
    .drop("rn")

    # metadado silver
    .withColumn("silver_processed_at", current_timestamp())
)

In [0]:
# Regra 1 — id_loja PK: sem nulos, sem duplicatas

resultado_pk = validate_key_columns(
    df          = df_silver,
    key_columns = KEY_COLUMNS
)
print(resultado_pk["message"])

In [0]:
# Regra 2 — cnpj deve ter 14 digitos numericos e ser unico

cnpj_invalido = df_silver.filter(
    col("cnpj").isNull() |
    (~col("cnpj").rlike(r"^\d{14}$"))
).count()

cnpj_duplicado = (
    df_silver
    .groupBy("cnpj")
    .count()
    .filter(col("count") > 1)
    .count()
)

if cnpj_invalido > 0:
    print(f"Atencao: {cnpj_invalido} registros com CNPJ fora do padrao de 14 digitos.")
    df_silver.filter(~col("cnpj").rlike(r"^\d{14}$")) \
        .select("id_loja", "nome_loja", "cnpj").show()
else:
    print("Validacao OK: todos os CNPJs tem 14 digitos.")

if cnpj_duplicado > 0:
    print(f"Atencao: {cnpj_duplicado} CNPJs duplicados encontrados.")
else:
    print("Validacao OK: nenhum CNPJ duplicado.")

In [0]:
# Regra 3 — estado_loja deve ser UF valida (2 letras)

uf_invalida = df_silver.filter(
    ~col("estado_loja").isin(UFS_VALIDAS)
).count()

if uf_invalida > 0:
    print(f"Atencao: {uf_invalida} registros com UF invalida.")
    df_silver.filter(
        ~col("estado_loja").isin(UFS_VALIDAS)
    ).select("id_loja", "nome_loja", "estado_loja").show()
else:
    print("Validacao OK: todos os estados sao UFs validas.")

In [0]:
# visualizar Silver

df_silver.printSchema()
total_silver = df_silver.count()

print(f"Total Bronze : {total_bronze:,}")
print(f"Total Silver : {total_silver:,}")
display(df_silver.limit(10))

In [0]:
# validar qualidade Silver

validate_silver_quality(
    df               = df_silver,
    required_columns = SILVER_REQUIRED_COLUMNS
)

In [0]:
# gravar Silver Delta no ADLS
# physical_lojas e dado estatico — sem particao

write_delta(
    df           = df_silver,
    path         = SILVER_PATH,
    mode         = SILVER_WRITE_MODE,
    partition_by = None,
    adls_options = adls_options
)

In [0]:
# ler Silver gravada para validacao

df_silver_saved = read_delta(
    spark        = spark,
    path         = SILVER_PATH,
    adls_options = adls_options
)

df_silver_saved.printSchema()
display(df_silver_saved.limit(10))

In [0]:
# validar Silver gravada

compare_row_counts(
    source_df = df_silver,
    target_df = df_silver_saved,
    label     = "Silver memoria x Silver gravada"
)

In [0]:
# validar schema final da Silver

colunas_ausentes = [
    c for c in SILVER_REQUIRED_COLUMNS
    if c not in df_silver_saved.columns
]

if colunas_ausentes:
    raise Exception(
        f"Colunas obrigatorias ausentes na Silver: {colunas_ausentes}"
    )

print("Validacao schema final Silver OK.")

In [0]:
# validar qualidade final Silver gravada

validate_silver_quality(
    df               = df_silver_saved,
    required_columns = SILVER_REQUIRED_COLUMNS
)

In [0]:
# Resumo

print("=" * 55)
print("SILVER physical_lojas concluida com sucesso!")
print("=" * 55)
print(f"""
   Bronze Path  : {BRONZE_PATH}
   Silver Path  : {SILVER_PATH}
   Total Bronze : {total_bronze:,}
   Total Silver : {total_silver:,}
   Particao     : sem particao (dado estatico)

Regras tecnicas aplicadas:
   Regra 1 : id_loja PK — sem nulos, sem duplicatas
   Regra 2 : cnpj 14 digitos e unico
   Regra 3 : estado_loja UF valida
   Regra 4 : nome_loja "nan" -> "Nao Informado"

   Status   : SUCESSO
""")